In [4]:
import arcpy
import os
import zipfile

# -------------------------------------------------------------
# ENVIRONMENT SETTINGS
# -------------------------------------------------------------
arcpy.env.addOutputsToMap = False
arcpy.env.overwriteOutput = True

# -------------------------------------------------------------
# USER CONFIGURATION
# -------------------------------------------------------------
kmz_file = r"C:\Users\gespi\Downloads\Amostras Estaleiro 01 KMZ.kmz"
output_directory = r"D:\KML_2_GDB"
output_gdb_name = "KML_Data_Converted"
match_table = r"memory\out_match_table"


# Establish Workspace Pathways
gdb_path = os.path.join(output_directory, f"{output_gdb_name}.gdb")
raw_kml_points = os.path.join(gdb_path, "Placemarks", "Points")
final_points_fc = os.path.join(gdb_path, "Field_Points_With_Photos")

# -------------------------------------------------------------
# STEP 1: CONVERT KMZ TO GEODATABASE
# -------------------------------------------------------------
print("Converting KMZ to File Geodatabase...")
arcpy.conversion.KMLToLayer(kmz_file, output_directory, output_gdb_name)

# -------------------------------------------------------------
# STEP 2: CLONE DATA TO BREAK LOCKS & ENABLE ATTACHMENTS
# -------------------------------------------------------------
print("Cloning KML points to a fresh feature class...")
arcpy.conversion.FeatureClassToFeatureClass(raw_kml_points, gdb_path, "Field_Points_With_Photos")

print("Assigning Global IDs...")
arcpy.management.AddGlobalIDs(final_points_fc)

print("Enabling attachments...")
arcpy.management.EnableAttachments(final_points_fc)

# -------------------------------------------------------------
# STEP 3: EXTRACT PHOTOS FROM KMZ
# -------------------------------------------------------------
print("Extracting images from the KMZ archive...")
extraction_dir = os.path.join(output_directory, "Extracted_KMZ_Contents")

with zipfile.ZipFile(kmz_file, 'r') as zip_ref:
    zip_ref.extractall(extraction_dir)

photo_folder = os.path.join(extraction_dir, "files")
if not os.path.exists(photo_folder):
    photo_folder = extraction_dir + "\images"

print(f"Targeting media directory: {photo_folder}")

# -------------------------------------------------------------
# STEP 4: ADD PHOTOS AS ATTACHMENTS
# -------------------------------------------------------------
print("Generating attachment match table...")

arcpy.management.GenerateAttachmentMatchTable(
    in_dataset=final_points_fc,
    in_folder=photo_folder,
    out_match_table= match_table,
    in_key_field="OBJECTID",
    in_file_filter="",
    in_use_relative_paths="RELATIVE",
    match_pattern="EXACT")

arcpy.management.GenerateAttachmentMatchTable(
    in_dataset=final_points_fc,
    in_folder=photo_folder,
    out_match_table=match_table ,
    in_key_field="OBJECTID",
    in_file_filter="",
    in_use_relative_paths="RELATIVE",
    match_pattern="EXACT")

print("Loading matches into Geodatabase Feature Class attachments...")

arcpy.management.AddAttachments(
    final_points_fc,
    "OBJECTID",
    match_table,
    "MatchID",
    "Filename",
    photo_folder)

print("\nWorkflow completed successfully! Look for 'Field_Points_With_Photos' in your GDB.")

Converting KMZ to File Geodatabase...
Cloning KML points to a fresh feature class...
Assigning Global IDs...
Enabling attachments...
Extracting images from the KMZ archive...
Targeting media directory: D:\KML_2_GDB\Extracted_KMZ_Contents\images
Generating attachment match table...
Loading matches into Geodatabase Feature Class attachments...

Workflow completed successfully! Look for 'Field_Points_With_Photos' in your GDB.
